# Mobile Market Intelligence & Price Prediction
**AICTE | IBM SkillsBuild Data Analytics with AI Internship 2026 | BharatCares**

**Student:** Aayush Gupta  
**Project:** Mobile Market Intelligence  
**Dataset:** Mobiles Dataset (2025).csv  
**Objective:** End-to-end data analytics and ML pipeline — data cleaning, EDA, visualization, price segmentation, correlation analysis, and regression-based price prediction for smartphone launch prices.

---

## Table of Contents
1. Environment Setup & Imports
2. Data Loading & Initial Inspection
3. Data Quality Audit
4. Data Cleaning Pipeline
5. Feature Engineering
6. Exploratory Data Analysis (EDA)
7. Market Analysis
8. Price Analysis & Segmentation
9. Correlation & Statistical Analysis
10. Brand Intelligence
11. Global Price Comparison
12. Machine Learning — Price Prediction
13. Model Evaluation & Diagnostics
14. Feature Importance
15. Prediction Demo
16. Automatic Insights
17. Summary

## 1. Environment Setup & Imports

In [ ]:
# Install required libraries (run once)
# !pip install pandas numpy matplotlib seaborn scikit-learn

import warnings
warnings.filterwarnings('ignore')

import re
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Display settings
pd.set_option('display.max_columns', 20)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

# Plot style
plt.rcParams.update({
    'figure.facecolor': '#0a0f1e',
    'axes.facecolor': '#111c33',
    'axes.edgecolor': '#1e2d4a',
    'axes.labelcolor': '#9aabbf',
    'xtick.color': '#9aabbf',
    'ytick.color': '#9aabbf',
    'text.color': '#e8edf5',
    'grid.color': '#1e2d4a',
    'grid.linestyle': '--',
    'grid.alpha': 0.5,
    'font.family': 'sans-serif',
    'font.size': 11,
})
COLORS = ['#3b82d4','#22d3ee','#a78bfa','#34d399','#f59e0b','#f87171','#fb923c','#e879f9',
          '#38bdf8','#4ade80','#facc15','#fb7185','#c084fc','#67e8f9']

print('All libraries imported successfully.')

## 2. Data Loading & Initial Inspection

In [ ]:
# Load raw dataset
DATA_PATH = Path('data/raw/Mobiles Dataset (2025).csv')
# Fallback: try current directory
if not DATA_PATH.exists():
    DATA_PATH = Path('Mobiles Dataset (2025).csv')

raw_df = pd.read_csv(DATA_PATH, encoding='utf-8', encoding_errors='replace')
print(f'Raw shape: {raw_df.shape}')
raw_df.head(5)

In [ ]:
print('=== COLUMN NAMES ===')
for i, col in enumerate(raw_df.columns):
    print(f'  {i+1:2d}. {col}')

In [ ]:
print('=== DATA TYPES ===')
print(raw_df.dtypes)
print(f'\nTotal rows    : {len(raw_df)}')
print(f'Total columns : {len(raw_df.columns)}')

In [ ]:
print('=== SAMPLE VALUES (first 3 rows) ===')
raw_df.iloc[:3].T

## 3. Data Quality Audit

In [ ]:
print('=== MISSING VALUES ===')
missing = raw_df.isnull().sum()
missing_pct = (missing / len(raw_df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0])

print('\n=== BLANK ROWS ===')
blank_rows = raw_df.isnull().all(axis=1).sum()
print(f'Blank rows: {blank_rows}')

In [ ]:
print('=== DUPLICATE RECORDS ===')
dup_key = raw_df['Company Name'].astype(str) + '|' + raw_df['Model Name'].astype(str)
n_dupes = dup_key.duplicated(keep='first').sum()
print(f'Duplicate model records: {n_dupes}')
if n_dupes > 0:
    dup_examples = raw_df[dup_key.duplicated(keep='first')][['Company Name','Model Name']].head(10)
    print('\nExample duplicates:')
    print(dup_examples.to_string())

In [ ]:
print('=== FORMAT INSPECTION ===')
sample_cols = ['RAM', 'Mobile Weight', 'Battery Capacity', 'Screen Size',
               'Front Camera', 'Back Camera',
               'Launched Price (India)', 'Launched Price (USA)']
for col in sample_cols:
    if col in raw_df.columns:
        samples = raw_df[col].dropna().unique()[:5]
        print(f'  {col}: {list(samples)}')

In [ ]:
print('=== UNIQUE BRANDS ===')
brands = raw_df['Company Name'].dropna().str.strip().unique()
print(f'Count: {len(brands)}')
print(sorted(brands))

In [ ]:
print('=== LAUNCH YEAR RANGE ===')
print(raw_df['Launched Year'].value_counts().sort_index())

## 4. Data Cleaning Pipeline

In [ ]:
# ============================================================
# FIELD PARSERS — reproducible transformations
# Raw columns are preserved; cleaned columns added alongside
# ============================================================

def parse_ram(val):
    """'6GB' -> 6.0 | '1.5GB' -> 1.5 | '8GB / 12GB' -> 8.0 (take lower)"""
    if pd.isna(val): return None
    nums = re.findall(r'\d+\.?\d*', str(val))
    return float(nums[0]) if nums else None

def parse_weight(val):
    """'174g' -> 174.0"""
    if pd.isna(val): return None
    nums = re.findall(r'\d+\.?\d*', str(val))
    return float(nums[0]) if nums else None

def parse_camera_primary(val):
    """'50MP + 12MP' -> 50.0 | 'Dual 32MP' -> 32.0 | '12MP / 4K' -> 12.0"""
    if pd.isna(val): return None
    v = re.sub(r'(?i)dual\s*', '', str(val))
    nums = re.findall(r'\d+\.?\d*', v)
    return float(nums[0]) if nums else None

def count_camera_lenses(val):
    """Count rear lenses via '+' separators"""
    if pd.isna(val): return 1
    return len(str(val).split('+'))

def parse_battery(val):
    """'3,600mAh' -> 3600"""
    if pd.isna(val): return None
    cleaned = re.sub(r'[^\d]', '', str(val))
    return int(cleaned) if cleaned else None

def parse_screen(val):
    """'6.1 inches' -> 6.1 | '6.7 inches (main), 2.7 inches (external)' -> 6.7"""
    if pd.isna(val): return None
    nums = re.findall(r'\d+\.?\d*', str(val))
    return float(nums[0]) if nums else None

def parse_india_price(val):
    """'INR 79,999' -> 79999 | 'INR 1,04,999' -> 104999"""
    if pd.isna(val): return None
    cleaned = re.sub(r'[^\d]', '', str(val))
    return float(cleaned) if cleaned else None

def parse_pakistan_price(val):
    """'PKR 224,999' -> 224999 | 'Not available' -> None"""
    if pd.isna(val): return None
    s = str(val).strip().lower()
    if s in ('not available', 'n/a', ''): return None
    cleaned = re.sub(r'[^\d]', '', str(val))
    return float(cleaned) if cleaned else None

def parse_china_price(val):
    """'CNY 5,799' -> 5799 | encoding errors -> None"""
    if pd.isna(val): return None
    cleaned = re.sub(r'[^\d,.]', '', str(val)).replace(',','').replace('.', '')
    return float(cleaned) if cleaned else None

def parse_usa_price(val):
    """'USD 799' -> 799 | 'USD 1,049' -> 1049 | 'USD 396,22' -> 396.22"""
    if pd.isna(val): return None
    s = re.sub(r'(?i)usd\s*', '', str(val)).strip()
    if re.match(r'^\d+,\d{2}$', s):  # European decimal
        s = s.replace(',', '.')
    else:
        s = s.replace(',', '')
    try: return float(s)
    except: return None

def parse_dubai_price(val):
    """'AED 2,799' -> 2799"""
    if pd.isna(val): return None
    cleaned = re.sub(r'[^\d.]', '', str(val))
    return float(cleaned) if cleaned else None

def assign_processor_family(processor):
    """Group 200+ processor strings into ~15 families"""
    if pd.isna(processor): return 'Unknown'
    p = str(processor).lower()
    if 'a1' in p and 'bionic' in p: return 'Apple'
    if 'a12z' in p or 'apple' in p: return 'Apple'
    if any(x in p for x in ['snapdragon 8 elite','snapdragon 8 gen 3','snapdragon 8 gen 2',
                             'snapdragon 8 gen 1','snapdragon 888','snapdragon 865','snapdragon 870',
                             'snapdragon 860','snapdragon 855','snapdragon 845','snapdragon 835',
                             'snapdragon 8+ gen']): return 'Snapdragon 8 (Flagship)'
    if 'snapdragon 8s' in p: return 'Snapdragon 8s (Near-Flagship)'
    if 'snapdragon 7' in p: return 'Snapdragon 7 (Upper-Mid)'
    if 'snapdragon 6' in p: return 'Snapdragon 6 (Mid)'
    if 'snapdragon 4' in p or re.search(r'snapdragon\s*(480|460|450|439|430|425)', p): return 'Snapdragon 4/5 (Entry-Mid)'
    if 'snapdragon' in p: return 'Snapdragon (Other)'
    if 'dimensity 9' in p: return 'Dimensity 9xxx (Flagship)'
    if 'dimensity 8' in p: return 'Dimensity 8xxx (Upper-Mid)'
    if 'dimensity 7' in p: return 'Dimensity 7xxx (Mid)'
    if any(x in p for x in ['dimensity 6','dimensity 1','dimensity 800','dimensity 900']): return 'Dimensity 6xx-1xxx (Entry-Mid)'
    if 'dimensity' in p: return 'Dimensity (Other)'
    if 'helio g9' in p: return 'Helio G9x (Mid-Budget)'
    if any(x in p for x in ['helio g8','helio g7']): return 'Helio G7-G8x (Budget)'
    if 'helio g' in p: return 'Helio G (Budget)'
    if any(x in p for x in ['helio p','helio a']): return 'Helio P/A (Budget)'
    if 'kirin 9000' in p or 'kirin 9010' in p: return 'Kirin 9000 (Flagship)'
    if 'kirin 9' in p: return 'Kirin 9xx (Flagship)'
    if any(x in p for x in ['kirin 8','kirin 7','kirin 710','kirin 820','kirin 985','kirin 990']): return 'Kirin 7xx-8xx (Mid)'
    if 'kirin' in p: return 'Kirin (Other)'
    if 'exynos 2' in p: return 'Exynos 2xxx (Flagship)'
    if 'exynos 1' in p: return 'Exynos 1xxx (Mid)'
    if re.search(r'exynos [789]', p): return 'Exynos 7xx-9xx (Budget-Mid)'
    if 'exynos' in p: return 'Exynos (Other)'
    if 'tensor' in p: return 'Google Tensor'
    if 'unisoc' in p or 'spreadtrum' in p: return 'Unisoc (Budget)'
    return 'Other'

def is_tablet(model_name, screen_size):
    keywords = ['ipad','tab ','galaxy tab','pad ','megapad','xpad','matepad']
    if any(k in str(model_name).lower() for k in keywords): return True
    if screen_size is not None and not math.isnan(float(screen_size or 0)) and float(screen_size) > 9.0: return True
    return False

def is_foldable(model_name):
    keywords = ['fold','flip','xs 2','open','magic v','magic vs','find n','mate x','razr','pocket']
    return any(k in str(model_name).lower() for k in keywords)

def assign_price_segment(price):
    if price is None or (isinstance(price, float) and math.isnan(price)): return 'Unknown'
    if price < 15000: return 'Budget'
    if price < 30000: return 'Mid-Range'
    if price < 60000: return 'Upper Mid-Range'
    if price < 100000: return 'Premium'
    return 'Ultra Premium'

print('All parsers defined.')

In [ ]:
# ============================================================
# APPLY CLEANING PIPELINE
# ============================================================
df = raw_df.copy()

# 1. Drop blank rows
df = df.dropna(how='all').reset_index(drop=True)

# 2. Normalise brand capitalisation
df['Company Name'] = df['Company Name'].str.strip()
df.loc[df['Company Name'] == 'Poco', 'Company Name'] = 'POCO'

# 3. Numeric spec columns
df['RAM_num']          = df['RAM'].apply(parse_ram)
df['Weight_num']       = df['Mobile Weight'].apply(parse_weight)
df['FrontCam_num']     = df['Front Camera'].apply(parse_camera_primary)
df['BackCam_primary']  = df['Back Camera'].apply(parse_camera_primary)
df['BackCam_lenses']   = df['Back Camera'].apply(count_camera_lenses)
df['Battery_num']      = df['Battery Capacity'].apply(parse_battery)
df['Screen_num']       = df['Screen Size'].apply(parse_screen)

# 4. Price columns
df['India_Price']    = df['Launched Price (India)'].apply(parse_india_price)
df['Pakistan_Price'] = df['Launched Price (Pakistan)'].apply(parse_pakistan_price)
df['China_Price']    = df['Launched Price (China)'].apply(parse_china_price)
df['USA_Price']      = df['Launched Price (USA)'].apply(parse_usa_price)
df['Dubai_Price']    = df['Launched Price (Dubai)'].apply(parse_dubai_price)

# 5. Flags
df['Is_Tablet']   = df.apply(lambda r: is_tablet(r['Model Name'], r['Screen_num']), axis=1)
df['Is_Foldable'] = df['Model Name'].apply(is_foldable)

# 6. Processor family
df['Processor_Family'] = df['Processor'].apply(assign_processor_family)

# 7. Price segment
df['Price_Segment'] = df['India_Price'].apply(assign_price_segment)

# 8. Deduplication
dup_key = df['Company Name'] + '|' + df['Model Name']
n_before = len(df)
df = df[~dup_key.duplicated(keep='first')].reset_index(drop=True)
n_dupes_removed = n_before - len(df)

print(f'Rows before dedup : {n_before}')
print(f'Duplicates removed: {n_dupes_removed}')
print(f'Clean records     : {len(df)}')
print(f'Unique brands     : {df["Company Name"].nunique()}')
print(f'Tablets flagged   : {df["Is_Tablet"].sum()}')
print(f'Foldables flagged : {df["Is_Foldable"].sum()}')
df.head(3)

## 5. Feature Engineering Summary

In [ ]:
engineered_cols = ['RAM_num','Weight_num','FrontCam_num','BackCam_primary','BackCam_lenses',
                   'Battery_num','Screen_num','India_Price','Pakistan_Price',
                   'China_Price','USA_Price','Dubai_Price',
                   'Is_Tablet','Is_Foldable','Processor_Family','Price_Segment']

print('=== ENGINEERED FEATURE STATS ===')
numeric_eng = [c for c in engineered_cols if df[c].dtype in [float, int, 'float64','int64']]
df[numeric_eng].describe().T.round(2)

In [ ]:
print('=== PRICE SEGMENT DISTRIBUTION ===')
seg_counts = df['Price_Segment'].value_counts()
print(seg_counts)
print(f'\nPrice segment thresholds (India ₹):')
print('  Budget         : < 15,000')
print('  Mid-Range      : 15,000 – 30,000')
print('  Upper Mid-Range: 30,000 – 60,000')
print('  Premium        : 60,000 – 1,00,000')
print('  Ultra Premium  : > 1,00,000')

## 6. Exploratory Data Analysis (EDA)

In [ ]:
# ── India Price Distribution ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0a0f1e')

prices = df['India_Price'].dropna()

# Histogram
axes[0].hist(prices, bins=30, color='#3b82d4', edgecolor='#0a0f1e', alpha=0.85)
axes[0].set_title('India Launch Price Distribution', color='#e8edf5', fontsize=13, fontweight='bold')
axes[0].set_xlabel('India Price (INR)', color='#9aabbf')
axes[0].set_ylabel('Number of Devices', color='#9aabbf')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x/1000:.0f}k'))
axes[0].axvline(prices.mean(), color='#22d3ee', linestyle='--', linewidth=1.5, label=f'Mean ₹{prices.mean()/1000:.0f}k')
axes[0].axvline(prices.median(), color='#f59e0b', linestyle='--', linewidth=1.5, label=f'Median ₹{prices.median()/1000:.0f}k')
axes[0].legend(facecolor='#111c33', edgecolor='#1e2d4a', labelcolor='#e8edf5')

# Box plot by segment
seg_order = ['Budget','Mid-Range','Upper Mid-Range','Premium','Ultra Premium']
seg_data = [df[df['Price_Segment']==s]['India_Price'].dropna().values for s in seg_order]
bp = axes[1].boxplot(seg_data, labels=seg_order, patch_artist=True, medianprops={'color':'#22d3ee','linewidth':2})
for patch, color in zip(bp['boxes'], COLORS):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].set_title('Price by Segment (Box Plot)', color='#e8edf5', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Price Segment', color='#9aabbf')
axes[1].set_ylabel('India Price (INR)', color='#9aabbf')
axes[1].set_xticklabels(seg_order, rotation=15, ha='right')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x/1000:.0f}k'))

plt.tight_layout()
plt.suptitle('India Launch Price Overview', color='#e8edf5', fontsize=15, y=1.02, fontweight='bold')
plt.show()

print(f'Mean   India Price : ₹{prices.mean():,.0f}')
print(f'Median India Price : ₹{prices.median():,.0f}')
print(f'Std Dev            : ₹{prices.std():,.0f}')
print(f'Min                : ₹{prices.min():,.0f}')
print(f'Max                : ₹{prices.max():,.0f}')

## 7. Market Analysis

In [ ]:
# ── Brand Distribution ────────────────────────────────────
brand_counts = df['Company Name'].value_counts().reset_index()
brand_counts.columns = ['Brand', 'Count']
brand_counts['Share %'] = (brand_counts['Count'] / len(df) * 100).round(2)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor('#0a0f1e')

# Bar chart — all brands
colors_bar = [COLORS[i % len(COLORS)] for i in range(len(brand_counts))]
axes[0].barh(brand_counts['Brand'][::-1], brand_counts['Count'][::-1], color=colors_bar[::-1])
axes[0].set_title('Models Per Brand', color='#e8edf5', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Number of Models', color='#9aabbf')

# Donut — top 10
top10 = brand_counts.head(10)
wedges, texts, autotexts = axes[1].pie(
    top10['Share %'], labels=top10['Brand'],
    colors=[COLORS[i % len(COLORS)] for i in range(10)],
    autopct='%1.1f%%', startangle=140,
    wedgeprops={'edgecolor': '#0a0f1e', 'linewidth': 1.5},
    pctdistance=0.75
)
for at in autotexts: at.set_color('#e8edf5'); at.set_fontsize(8)
for t in texts: t.set_color('#e8edf5'); t.set_fontsize(9)

# Draw donut hole
centre = plt.Circle((0,0), 0.5, fc='#0a0f1e')
axes[1].add_patch(centre)
axes[1].set_title('Market Share — Top 10 Brands', color='#e8edf5', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()
print(brand_counts.to_string(index=False))

In [ ]:
# ── Yearly Launches & Price Trend ────────────────────────
yearly = df.groupby('Launched Year').agg(
    count=('Model Name', 'count'),
    avg_india_price=('India_Price', 'mean')
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0a0f1e')

axes[0].bar(yearly['Launched Year'], yearly['count'], color='#3b82d4', edgecolor='#0a0f1e')
axes[0].set_title('Device Launches per Year', color='#e8edf5', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Year', color='#9aabbf')
axes[0].set_ylabel('Number of Models', color='#9aabbf')

axes[1].plot(yearly['Launched Year'], yearly['avg_india_price']/1000, marker='o',
             color='#22d3ee', linewidth=2.5, markersize=6)
axes[1].fill_between(yearly['Launched Year'], yearly['avg_india_price']/1000,
                     alpha=0.15, color='#22d3ee')
axes[1].set_title('Avg India Launch Price by Year', color='#e8edf5', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Year', color='#9aabbf')
axes[1].set_ylabel('Avg Price (₹ thousands)', color='#9aabbf')

plt.tight_layout()
plt.show()
print(yearly.to_string(index=False))

In [ ]:
# ── RAM Distribution ──────────────────────────────────────
ram_dist = df['RAM_num'].dropna().astype(int).value_counts().sort_index()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0a0f1e')

axes[0].bar(ram_dist.index.astype(str), ram_dist.values, color='#a78bfa', edgecolor='#0a0f1e')
axes[0].set_title('RAM Distribution', color='#e8edf5', fontsize=13, fontweight='bold')
axes[0].set_xlabel('RAM (GB)', color='#9aabbf')
axes[0].set_ylabel('Number of Models', color='#9aabbf')

ram_year = df.groupby('Launched Year')['RAM_num'].mean()
axes[1].plot(ram_year.index, ram_year.values, marker='s', color='#a78bfa', linewidth=2.5, markersize=6)
axes[1].fill_between(ram_year.index, ram_year.values, alpha=0.15, color='#a78bfa')
axes[1].set_title('Avg RAM by Launch Year', color='#e8edf5', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Year', color='#9aabbf')
axes[1].set_ylabel('Avg RAM (GB)', color='#9aabbf')

plt.tight_layout()
plt.show()

In [ ]:
# ── Battery Distribution ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0a0f1e')

bat = df['Battery_num'].dropna()
axes[0].hist(bat, bins=25, color='#34d399', edgecolor='#0a0f1e', alpha=0.85)
axes[0].set_title('Battery Capacity Distribution', color='#e8edf5', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Battery (mAh)', color='#9aabbf')
axes[0].set_ylabel('Count', color='#9aabbf')

avg_bat = df.groupby('Company Name')['Battery_num'].mean().sort_values(ascending=False).head(15)
axes[1].barh(avg_bat.index[::-1], avg_bat.values[::-1],
             color=[COLORS[i % len(COLORS)] for i in range(15)])
axes[1].set_title('Avg Battery per Brand (Top 15)', color='#e8edf5', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Avg Battery (mAh)', color='#9aabbf')

plt.tight_layout()
plt.show()

## 8. Price Analysis & Segmentation

In [ ]:
# ── Price Segment Analysis ────────────────────────────────
seg_order = ['Budget','Mid-Range','Upper Mid-Range','Premium','Ultra Premium']
seg_stats = df[df['Price_Segment'].isin(seg_order)].groupby('Price_Segment').agg(
    Count=('India_Price','count'),
    Avg_Price=('India_Price','mean'),
    Median_Price=('India_Price','median'),
    Avg_RAM=('RAM_num','mean'),
    Avg_Battery=('Battery_num','mean'),
    Avg_Screen=('Screen_num','mean'),
).reindex(seg_order).round(2)

print('=== PRICE SEGMENT STATISTICS ===')
print(seg_stats.to_string())

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor('#0a0f1e')

# Count
counts = seg_stats['Count']
axes[0].bar(seg_order, counts, color=COLORS[:5], edgecolor='#0a0f1e')
axes[0].set_title('Devices per Segment', color='#e8edf5', fontsize=12, fontweight='bold')
axes[0].set_xticklabels(seg_order, rotation=15, ha='right', fontsize=9)

# Avg price
axes[1].bar(seg_order, seg_stats['Avg_Price']/1000, color=COLORS[:5], edgecolor='#0a0f1e')
axes[1].set_title('Avg India Price per Segment', color='#e8edf5', fontsize=12, fontweight='bold')
axes[1].set_ylabel('₹ thousands', color='#9aabbf')
axes[1].set_xticklabels(seg_order, rotation=15, ha='right', fontsize=9)

# Avg RAM
axes[2].bar(seg_order, seg_stats['Avg_RAM'], color=COLORS[:5], edgecolor='#0a0f1e')
axes[2].set_title('Avg RAM per Segment (GB)', color='#e8edf5', fontsize=12, fontweight='bold')
axes[2].set_ylabel('Avg RAM (GB)', color='#9aabbf')
axes[2].set_xticklabels(seg_order, rotation=15, ha='right', fontsize=9)

plt.tight_layout()
plt.show()

## 9. Correlation & Statistical Analysis

In [ ]:
# ── Correlation Matrix ────────────────────────────────────
num_cols = ['India_Price','RAM_num','Battery_num','Screen_num',
            'FrontCam_num','BackCam_primary','Weight_num','Launched Year']
corr_df = df[num_cols].dropna().corr().round(3)

labels = ['India Price','RAM','Battery','Screen',
          'Front Cam','Back Cam','Weight','Year']

fig, ax = plt.subplots(figsize=(10, 8))
fig.patch.set_facecolor('#0a0f1e')
sns.heatmap(
    corr_df, annot=True, fmt='.2f', cmap='coolwarm',
    xticklabels=labels, yticklabels=labels,
    ax=ax, linewidths=0.5, linecolor='#0a0f1e',
    annot_kws={'size': 9}, vmin=-1, vmax=1
)
ax.set_title('Pearson Correlation Matrix', color='#e8edf5', fontsize=14, fontweight='bold', pad=15)
ax.tick_params(colors='#9aabbf')
plt.tight_layout()
plt.show()

print('\nCorrelation with India Price (sorted):')
india_corr = corr_df['India_Price'].drop('India_Price').sort_values(ascending=False)
for feat, corr_val in india_corr.items():
    print(f'  {feat:20s}: {corr_val:+.3f}')
print('\n⚠ Correlation measures linear association only. Correlation ≠ causation.')

In [ ]:
# ── Scatter Plots: Spec vs India Price ───────────────────
specs = [
    ('RAM_num',        'RAM (GB)'),
    ('Battery_num',    'Battery (mAh)'),
    ('Screen_num',     'Screen (inches)'),
    ('BackCam_primary','Back Camera (MP)'),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.patch.set_facecolor('#0a0f1e')
axes = axes.flatten()

for i, (col, label) in enumerate(specs):
    sub = df[['India_Price', col]].dropna().sample(min(500, len(df)), random_state=42)
    r = sub['India_Price'].corr(sub[col])
    axes[i].scatter(sub[col], sub['India_Price']/1000, alpha=0.45,
                    color=COLORS[i], s=20, edgecolors='none')
    axes[i].set_xlabel(label, color='#9aabbf')
    axes[i].set_ylabel('India Price (₹ thousands)', color='#9aabbf')
    axes[i].set_title(f'{label} vs India Price  (r = {r:.3f})',
                      color='#e8edf5', fontsize=11, fontweight='bold')

plt.suptitle('Spec vs India Launch Price — Scatter Plots', color='#e8edf5', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 10. Brand Intelligence

In [ ]:
# ── Brand Profile Table ───────────────────────────────────
brand_profile = df.groupby('Company Name').agg(
    Models=('Model Name','count'),
    Avg_Price=('India_Price','mean'),
    Median_Price=('India_Price','median'),
    Min_Price=('India_Price','min'),
    Max_Price=('India_Price','max'),
    Avg_RAM=('RAM_num','mean'),
    Avg_Battery=('Battery_num','mean'),
    Avg_Screen=('Screen_num','mean'),
).round(2).sort_values('Models', ascending=False)

print('=== BRAND PROFILE ===')
brand_profile

In [ ]:
# ── Brand Avg Price Comparison ────────────────────────────
top_brands = brand_profile[brand_profile['Models'] >= 5].sort_values('Avg_Price', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor('#0a0f1e')

colors_b = [COLORS[i % len(COLORS)] for i in range(len(top_brands))]
axes[0].barh(top_brands.index[::-1], top_brands['Avg_Price'][::-1]/1000, color=colors_b[::-1])
axes[0].set_title('Avg India Price by Brand (≥5 models)', color='#e8edf5', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Avg India Price (₹ thousands)', color='#9aabbf')

axes[1].barh(top_brands.index[::-1], top_brands['Avg_RAM'][::-1],
             color=[COLORS[(i+3) % len(COLORS)] for i in range(len(top_brands))][::-1])
axes[1].set_title('Avg RAM by Brand (≥5 models)', color='#e8edf5', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Avg RAM (GB)', color='#9aabbf')

plt.tight_layout()
plt.show()

In [ ]:
# ── Brand Segment Heatmap ─────────────────────────────────
seg_brand = df.groupby(['Company Name','Price_Segment']).size().unstack(fill_value=0)
seg_brand = seg_brand.reindex(columns=['Budget','Mid-Range','Upper Mid-Range','Premium','Ultra Premium'], fill_value=0)
seg_brand = seg_brand[seg_brand.sum(axis=1) >= 5].sort_values('Premium', ascending=False)

fig, ax = plt.subplots(figsize=(12, max(5, len(seg_brand)*0.4 + 2)))
fig.patch.set_facecolor('#0a0f1e')
sns.heatmap(seg_brand, annot=True, fmt='d', cmap='Blues',
            ax=ax, linewidths=0.5, linecolor='#0a0f1e',
            annot_kws={'size': 9})
ax.set_title('Brand × Price Segment Distribution (model count)', color='#e8edf5', fontsize=13, fontweight='bold')
ax.set_xlabel('Price Segment', color='#9aabbf')
ax.set_ylabel('Brand', color='#9aabbf')
ax.tick_params(colors='#9aabbf')
plt.tight_layout()
plt.show()

## 11. Global Price Comparison

In [ ]:
# ── Dataset-Wide Country Averages ─────────────────────────
country_avgs = {
    'India (INR)': df['India_Price'].mean(),
    'Pakistan (PKR)': df['Pakistan_Price'].mean(),
    'China (CNY)': df['China_Price'].mean(),
    'USA (USD)': df['USA_Price'].mean(),
    'Dubai (AED)': df['Dubai_Price'].mean(),
}
print('=== AVG LAUNCH PRICES PER COUNTRY (native currencies) ===')
for k, v in country_avgs.items():
    print(f'  {k:20s}: {v:,.2f}')
print('\n⚠ Prices are in native currencies. Do not compare values directly without exchange rates.')

In [ ]:
# ── Example: Multi-country price for specific models ─────
sample_models = ['iPhone 16 128GB', 'Samsung Galaxy S24 128GB', 'OnePlus 12 256GB',
                 'Google Pixel 9 128GB', 'Sony Xperia 1 VI 256GB']

price_cols = ['India_Price','Pakistan_Price','China_Price','USA_Price','Dubai_Price']
col_labels = ['India (INR)','Pakistan (PKR)','China (CNY)','USA (USD)','Dubai (AED)']

rows = []
for m in sample_models:
    row = df[df['Model Name'].str.lower() == m.lower()]
    if row.empty:
        row = df[df['Model Name'].str.lower().str.contains(m.lower(), regex=False, na=False)]
    if not row.empty:
        r = row.iloc[0]
        rows.append([r['Company Name'], r['Model Name']] + [r[c] for c in price_cols])

global_df = pd.DataFrame(rows, columns=['Brand','Model'] + col_labels)
print(global_df.to_string(index=False))

## 12. Machine Learning — Price Prediction

In [ ]:
# ── Feature & Target Preparation ─────────────────────────
NUMERIC_FEATURES = ['RAM_num','Battery_num','FrontCam_num','BackCam_primary',
                    'BackCam_lenses','Screen_num','Weight_num','Launched Year']
CAT_FEATURES = ['Company Name','Processor_Family']
TARGET = 'India_Price'

# Build clean ML subset
ml_df = df[NUMERIC_FEATURES + CAT_FEATURES + [TARGET]].dropna(subset=[TARGET] + NUMERIC_FEATURES).copy()
for c in CAT_FEATURES:
    ml_df[c] = ml_df[c].fillna('Unknown')

X = ml_df[NUMERIC_FEATURES + CAT_FEATURES]
y = ml_df[TARGET].astype(float)

print(f'Feature matrix shape : {X.shape}')
print(f'Target vector shape  : {y.shape}')
print(f'Target mean          : ₹{y.mean():,.0f}')
print(f'Target std           : ₹{y.std():,.0f}')

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'\nTrain size : {len(X_train)}')
print(f'Test size  : {len(X_test)}')

In [ ]:
# ── Build Preprocessing Pipeline ─────────────────────────
preprocessor = ColumnTransformer(transformers=[
    ('num', 'passthrough', NUMERIC_FEATURES),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CAT_FEATURES),
])

# Helper
def evaluate(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = math.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    print(f'  {name:28s}  MAE=₹{mae:>10,.0f}  RMSE=₹{rmse:>10,.0f}  R²={r2:.4f}')
    return {'model': name, 'mae': round(mae, 2), 'rmse': round(rmse, 2), 'r2': round(r2, 4)}

results = []

# Baseline
train_mean = float(y_train.mean())
y_pred_base = np.full(len(y_test), train_mean)
print('=== MODEL EVALUATION ON TEST SET (80/20 split) ===')
results.append(evaluate('Baseline (Mean)', y_test, y_pred_base))

# Model definitions
models_def = [
    ('Linear Regression',    LinearRegression()),
    ('Decision Tree',        DecisionTreeRegressor(max_depth=8, random_state=42)),
    ('Random Forest',        RandomForestRegressor(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1)),
    ('Gradient Boosting',    GradientBoostingRegressor(n_estimators=200, max_depth=5, learning_rate=0.05, random_state=42)),
]

trained_models = {}
for name, estimator in models_def:
    pipe = Pipeline([('preprocessor', preprocessor), ('model', estimator)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    results.append(evaluate(name, y_test, y_pred))
    trained_models[name] = pipe

metrics_df = pd.DataFrame(results)
print('\n=== METRICS TABLE ===')
print(metrics_df.to_string(index=False))

## 13. Model Evaluation & Diagnostics

In [ ]:
# ── Metrics Bar Charts ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor('#0a0f1e')

model_names = metrics_df['model']
bar_colors = [COLORS[i % len(COLORS)] for i in range(len(model_names))]

for ax, metric, title in zip(axes, ['mae','rmse','r2'],
                              ['MAE (₹) — lower is better',
                               'RMSE (₹) — lower is better',
                               'R² — higher is better']):
    ax.bar(model_names, metrics_df[metric], color=bar_colors, edgecolor='#0a0f1e')
    ax.set_title(title, color='#e8edf5', fontsize=11, fontweight='bold')
    ax.set_xticklabels(model_names, rotation=20, ha='right', fontsize=8)

plt.suptitle('Model Performance Comparison', color='#e8edf5', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Actual vs Predicted — Random Forest ──────────────────
rf_pipe = trained_models['Random Forest']
y_pred_rf = rf_pipe.predict(X_test)
residuals = y_test.values - y_pred_rf

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0a0f1e')

# Actual vs Predicted
axes[0].scatter(y_test/1000, y_pred_rf/1000, alpha=0.45, color='#3b82d4', s=15, edgecolors='none')
mn, mx = min(y_test.min(), y_pred_rf.min())/1000, max(y_test.max(), y_pred_rf.max())/1000
axes[0].plot([mn, mx], [mn, mx], color='#22d3ee', linestyle='--', linewidth=1.5)
axes[0].set_xlabel('Actual India Price (₹k)', color='#9aabbf')
axes[0].set_ylabel('Predicted India Price (₹k)', color='#9aabbf')
axes[0].set_title('Actual vs Predicted — Random Forest', color='#e8edf5', fontsize=12, fontweight='bold')

# Residual distribution
axes[1].hist(residuals/1000, bins=40, color='#a78bfa', edgecolor='#0a0f1e', alpha=0.85)
axes[1].axvline(0, color='#22d3ee', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Residual (₹k)', color='#9aabbf')
axes[1].set_ylabel('Count', color='#9aabbf')
axes[1].set_title('Residual Distribution', color='#e8edf5', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print(f'Residuals — Mean: ₹{residuals.mean():,.0f}  Std: ₹{residuals.std():,.0f}')

## 14. Feature Importance

In [ ]:
# ── Random Forest Feature Importance ─────────────────────
rf_est = rf_pipe.named_steps['model']
ohe = rf_pipe.named_steps['preprocessor'].named_transformers_['cat']
cat_names = list(ohe.get_feature_names_out(CAT_FEATURES))
all_feature_names = NUMERIC_FEATURES + cat_names

importances = rf_est.feature_importances_
feat_imp = pd.Series(importances, index=all_feature_names).sort_values(ascending=False)

top_n = 20
top_feats = feat_imp.head(top_n)

fig, ax = plt.subplots(figsize=(10, 8))
fig.patch.set_facecolor('#0a0f1e')
colors_fi = ['#22d3ee' if i < 3 else '#3b82d4' if i < 8 else '#1e2d4a' for i in range(top_n)]
ax.barh(top_feats.index[::-1], top_feats.values[::-1], color=colors_fi[::-1])
ax.set_title(f'Top {top_n} Feature Importances — Random Forest\n(Gini impurity-based)',
             color='#e8edf5', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score', color='#9aabbf')
plt.tight_layout()
plt.show()

print('\n=== TOP 10 FEATURES ===')
for feat, imp in feat_imp.head(10).items():
    print(f'  {feat:45s}: {imp:.5f}')
print('\nNote: Feature importance reflects statistical association in this dataset, not causal contribution.')

## 15. Prediction Demo

In [ ]:
# ── Predict Price for New Specifications ─────────────────
# Uses the trained Random Forest pipeline
# ⚠ Disclaimer: estimate based on dataset patterns only — not a market quotation

def predict_india_price(
    company, ram_gb, battery_mah, front_cam_mp, back_cam_mp,
    back_cam_lenses, screen_inches, weight_g, launched_year, processor_family,
    model_name='Random Forest'
):
    pipe = trained_models[model_name]
    x = pd.DataFrame([{
        'RAM_num': ram_gb, 'Battery_num': battery_mah,
        'FrontCam_num': front_cam_mp, 'BackCam_primary': back_cam_mp,
        'BackCam_lenses': back_cam_lenses, 'Screen_num': screen_inches,
        'Weight_num': weight_g, 'Launched Year': launched_year,
        'Company Name': company, 'Processor_Family': processor_family,
    }])
    pred = float(pipe.predict(x)[0])
    y_pred_test = pipe.predict(X_test)
    std = float(np.std(y_test.values - y_pred_test))
    return {
        'estimated_price': round(pred),
        'uncertainty_low': round(max(0, pred - std)),
        'uncertainty_high': round(pred + std),
        'model': model_name,
    }

# === DEMO PREDICTIONS ===
test_cases = [
    dict(company='Samsung', ram_gb=8, battery_mah=5000, front_cam_mp=12, back_cam_mp=50,
         back_cam_lenses=2, screen_inches=6.5, weight_g=190, launched_year=2024,
         processor_family='Snapdragon 8 (Flagship)'),
    dict(company='Apple', ram_gb=6, battery_mah=3600, front_cam_mp=12, back_cam_mp=48,
         back_cam_lenses=1, screen_inches=6.1, weight_g=174, launched_year=2024,
         processor_family='Apple'),
    dict(company='Realme', ram_gb=6, battery_mah=5000, front_cam_mp=16, back_cam_mp=50,
         back_cam_lenses=2, screen_inches=6.6, weight_g=190, launched_year=2024,
         processor_family='Helio G9x (Mid-Budget)'),
    dict(company='OnePlus', ram_gb=12, battery_mah=5000, front_cam_mp=32, back_cam_mp=50,
         back_cam_lenses=2, screen_inches=6.7, weight_g=205, launched_year=2024,
         processor_family='Snapdragon 8 (Flagship)'),
]

print('=== PRICE PREDICTION DEMO ===')
for tc in test_cases:
    result = predict_india_price(**tc)
    print(f"  {tc['company']:12s} {tc['ram_gb']}GB RAM  {tc['processor_family'][:30]:30s}")
    print(f"    Estimated: ₹{result['estimated_price']:>8,}  ")
    print(f"    Range    : ₹{result['uncertainty_low']:>8,} – ₹{result['uncertainty_high']:>8,}")
    print()

print('⚠ DISCLAIMER: These are machine-learning estimates based on patterns in the provided')
print('  dataset and should not be treated as actual market quotations.')

## 16. Automatic Insights

In [ ]:
# ── Auto-generate insights from actual data ───────────────
# All values computed from the cleaned dataset

insights = []

# 1. Top brand
top_brand = df['Company Name'].value_counts().idxmax()
top_brand_pct = round(df['Company Name'].value_counts().iloc[0] / len(df) * 100, 1)
insights.append(f'[Market]       {top_brand} has the most models ({top_brand_pct}% of the dataset).')

# 2. Median price
med_price = int(df['India_Price'].dropna().median())
insights.append(f'[Pricing]      Median India launch price: ₹{med_price:,}')

# 3. Most common RAM
common_ram = int(df['RAM_num'].dropna().mode()[0])
ram_pct = round((df['RAM_num'] == common_ram).sum() / len(df) * 100, 1)
insights.append(f'[Spec]         {common_ram}GB is the most common RAM ({ram_pct}% of devices).')

# 4. Price trend
p2020 = df[df['Launched Year'] == 2020]['India_Price'].mean()
latest_yr = int(df['Launched Year'].max())
p_latest = df[df['Launched Year'] == latest_yr]['India_Price'].mean()
pct_chg = round((p_latest - p2020) / p2020 * 100, 1)
direction = 'increased' if pct_chg > 0 else 'decreased'
insights.append(f'[Trend]        Avg India price {direction} {abs(pct_chg)}% from 2020 (₹{int(p2020):,}) to {latest_yr} (₹{int(p_latest):,}).')

# 5. Battery-price correlation
r_bat = df[['Battery_num','India_Price']].dropna().corr().iloc[0,1]
insights.append(f'[Correlation]  Battery vs India Price: r = {r_bat:.3f} (correlation ≠ causation)')

# 6. RAM-price correlation
r_ram = df[['RAM_num','India_Price']].dropna().corr().iloc[0,1]
insights.append(f'[Correlation]  RAM vs India Price: r = {r_ram:.3f}')

# 7. Top processor family
top_proc = df['Processor_Family'].value_counts().idxmax()
top_proc_pct = round(df['Processor_Family'].value_counts().iloc[0] / len(df) * 100, 1)
insights.append(f'[Spec]         "{top_proc}" is the most common processor family ({top_proc_pct}%).')

# 8. Budget vs Ultra Premium
budget_pct = round((df['Price_Segment']=='Budget').sum() / len(df) * 100, 1)
ultra_pct  = round((df['Price_Segment']=='Ultra Premium').sum() / len(df) * 100, 1)
insights.append(f'[Market]       Budget (<₹15k): {budget_pct}% | Ultra Premium (>₹1L): {ultra_pct}%')

# 9. Most active year
top_yr = int(df['Launched Year'].value_counts().idxmax())
top_yr_cnt = int(df['Launched Year'].value_counts().iloc[0])
insights.append(f'[Trend]        Most active launch year: {top_yr} ({top_yr_cnt} devices).')

# 10. RF R²
rf_metrics = next(m for m in results if m['model'] == 'Random Forest')
insights.append(f'[ML]           Random Forest R²={rf_metrics["r2"]} MAE=₹{rf_metrics["mae"]:,.0f} on test set.')

print('=== AUTO-GENERATED INSIGHTS FROM DATASET ===')
for ins in insights:
    print(f'  • {ins}')
print('\n⚠ All values are computed from the actual cleaned dataset. No values are fabricated.')

## 17. Summary

In [ ]:
print('='*60)
print(' MOBILE MARKET INTELLIGENCE — PROJECT SUMMARY')
print('='*60)
print(f'  Dataset             : Mobiles Dataset (2025).csv')
print(f'  Raw records         : {len(raw_df)}')
print(f'  Clean records       : {len(df)}')
print(f'  Duplicates removed  : {n_dupes_removed}')
print(f'  Unique brands       : {df["Company Name"].nunique()}')
print(f'  Year range          : {int(df["Launched Year"].min())} – {int(df["Launched Year"].max())}')
print(f'  Tablets flagged     : {int(df["Is_Tablet"].sum())}')
print(f'  Foldables flagged   : {int(df["Is_Foldable"].sum())}')
print()
print(f'  Avg India Price     : ₹{df["India_Price"].mean():,.0f}')
print(f'  Median India Price  : ₹{df["India_Price"].median():,.0f}')
print()
print('  ML Model Results (Random Forest):')
print(f'    R²   = {rf_metrics["r2"]}')
print(f'    MAE  = ₹{rf_metrics["mae"]:,.0f}')
print(f'    RMSE = ₹{rf_metrics["rmse"]:,.0f}')
print()
print('  Key Findings:')
for ins in insights:
    print(f'    • {ins}')
print()
print('  Student : Aayush Gupta')
print('  Program : AICTE | IBM SkillsBuild Data Analytics with AI Internship 2026')
print('='*60)